# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.ensemble import VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from joblib import dump


## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv('../../data/day-of-week-not-scaled.csv')

old_df = pd.read_csv('../../data/dayofweek.csv')
df['dayofweek'] = old_df['dayofweek']

X = df.drop('dayofweek', axis=1)
y = df['dayofweek']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=21, stratify=y_train)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [3]:
svc = SVC(random_state=21,
          kernel='rbf',
          gamma='auto',
          C=10.0,
          class_weight='balanced',
          probability=True)
svc.fit(X_train, y_train)
predict = svc.predict(X_valid)

acc = accuracy_score(y_valid, predict)
prec = precision_score(y_valid, predict, average="weighted")
rec = recall_score(y_valid, predict, average="weighted")

for name, value in [
    ("accuracy", acc),
    ("precision", prec),
    ("recall", rec)
]:
    print(f"{name} is {value:.5}")

accuracy is 0.85926
precision is 0.86352
recall is 0.85926


In [4]:
tree = DecisionTreeClassifier(random_state=21,
                            class_weight=None,
                            criterion='gini',
                            max_depth=25,
                            )
tree.fit(X_train, y_train)
predict = tree.predict(X_valid)

acc = accuracy_score(y_valid, predict)
prec = precision_score(y_valid, predict, average="weighted")
rec = recall_score(y_valid, predict, average="weighted")

for name, value in [
    ("accuracy", acc),
    ("precision", prec),
    ("recall", rec)
]:
    print(f"{name} is {value:.5}")

accuracy is 0.86296
precision is 0.86376
recall is 0.86296


In [5]:
forest = RandomForestClassifier(random_state=21,
                            n_estimators=100,
                            max_depth=49, 
                            class_weight=None,
                            criterion='gini')
forest.fit(X_train, y_train)
predict = forest.predict(X_valid)

acc = accuracy_score(y_valid, predict)
prec = precision_score(y_valid, predict, average="weighted")
rec = recall_score(y_valid, predict, average="weighted")

for name, value in [
    ("accuracy", acc),
    ("precision", prec),
    ("recall", rec)
]:
    print(f"{name} is {value:.5}")

accuracy is 0.8963
precision is 0.89588
recall is 0.8963


## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

## Default parameters

In [6]:
vc = VotingClassifier(
    estimators=[
        ('svc', svc),
        ('tree', tree),
        ('forest', forest)
    ], 
)

vc.fit(X_train, y_train)

valid_pred = vc.predict(X_valid)  

accuracy = accuracy_score(y_valid, valid_pred)
precision = precision_score(y_valid, valid_pred, average='weighted') 
recall = recall_score(y_valid, valid_pred, average='weighted')       

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)

Accuracy: 0.9037037037037037
Precision: 0.9046182784871126
Recall: 0.9037037037037037


## The best parameters

In [7]:
vc = VotingClassifier(
    estimators=[
        ('svc', svc),
        ('tree', tree),
        ('forest', forest)
    ],
    voting='soft',
    weights=[3,1,3]
)

vc.fit(X_train, y_train)

valid_pred = vc.predict(X_valid)  

accuracy = accuracy_score(y_valid, valid_pred)
precision = precision_score(y_valid, valid_pred, average='weighted') 
recall = recall_score(y_valid, valid_pred, average='weighted')       

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)

Accuracy: 0.9148148148148149
Precision: 0.9165520708169186
Recall: 0.9148148148148149


## The best parameters with test dataset

In [8]:
vc = VotingClassifier(
    estimators=[
        ('svc', svc),
        ('tree', tree),
        ('forest', forest)
    ],
    voting='soft',
    weights=[3,1,3]
)

vc.fit(X_train, y_train)

test_pred = vc.predict(X_test)  

accuracy = accuracy_score(y_test, test_pred)
precision = precision_score(y_test, test_pred, average='weighted') 
recall = recall_score(y_test, test_pred, average='weighted')       

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)

Accuracy: 0.8994082840236687
Precision: 0.9021109846892548
Recall: 0.8994082840236687


## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

## Default parameters

In [9]:
bagged_svc = BaggingClassifier(
    estimator=svc,
    n_jobs=-1,
    random_state=21
)

bagged_svc.fit(X_train, y_train)
valid_pred = bagged_svc.predict(X_valid)  

accuracy = accuracy_score(y_valid, valid_pred)
precision = precision_score(y_valid, valid_pred, average='weighted') 
recall = recall_score(y_valid, valid_pred, average='weighted')       

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)

Accuracy: 0.8851851851851852
Precision: 0.8914269078037195
Recall: 0.8851851851851852


## The best parameters

In [10]:
bagged_svc = BaggingClassifier(
    estimator=svc,
    n_estimators=42,
    bootstrap=True,
    n_jobs=-1,
    random_state=21
)

bagged_svc.fit(X_train, y_train)
valid_pred = bagged_svc.predict(X_valid)  

accuracy = accuracy_score(y_valid, valid_pred)
precision = precision_score(y_valid, valid_pred, average='weighted') 
recall = recall_score(y_valid, valid_pred, average='weighted')       

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)


Accuracy: 0.8888888888888888
Precision: 0.8931108870834898
Recall: 0.8888888888888888


## The best parameters with test dataset

In [11]:
bagged_svc = BaggingClassifier(
    estimator=svc,
    n_estimators=42,
    bootstrap=True,
    n_jobs=-1,
    random_state=21
)

bagged_svc.fit(X_train, y_train)
test_pred = bagged_svc.predict(X_test)  

accuracy = accuracy_score(y_test, test_pred)
precision = precision_score(y_test, test_pred, average='weighted') 
recall = recall_score(y_test, test_pred, average='weighted')       

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)


Accuracy: 0.8668639053254438
Precision: 0.8689904260196039
Recall: 0.8668639053254438


## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [12]:

best_params = None
best_acc = -999
best_prec = -999

n_splits_list = [2, 3, 4, 5, 6, 7]
passthrough_list = [False, True]
res = []

for n_splits in n_splits_list:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=21)

    for passthrough in passthrough_list:
        stack_clf = StackingClassifier(
            estimators=[
                ('svc', svc),
                ('tree', tree),
                ('forest', forest)
            ],

            final_estimator=OneVsRestClassifier(
                LogisticRegression(solver='liblinear')
            ),
            cv=cv,
            passthrough=passthrough
        )

        stack_clf.fit(X_train, y_train)

        y_val_pred = stack_clf.predict(X_valid)

        acc = accuracy_score(y_valid, y_val_pred)
        prec = precision_score(y_valid, y_val_pred, average='macro', zero_division=0)
        rec = recall_score(y_valid, y_val_pred, average='macro', zero_division=0)

        res.append({
            'n_splits': n_splits,
            'passthrough': passthrough,
            'accuracy': acc,
            'precision': prec,
            'recall': rec
        })


    if (acc > best_acc) or (acc == best_acc and prec > best_prec):
        best_acc = acc
        best_prec = prec
        best_params = {
            'n_splits': n_splits,
            'passthrough': passthrough
        }


print(best_params)
print("Validation accuracy:", best_acc)
print("Validation Precision:", best_prec)

{'n_splits': 3, 'passthrough': True}
Validation accuracy: 0.9074074074074074
Validation Precision: 0.9221231674365381


In [13]:
best_cv = StratifiedKFold(
    n_splits=best_params['n_splits'],
    shuffle=True,
    random_state=21
)

best_stack_clf = StackingClassifier(
    estimators=[
        ('svc', svc),
        ('tree', tree),
        ('forest', forest)
    ],
    final_estimator=OneVsRestClassifier(
        LogisticRegression(solver='liblinear')
    ),
    cv=best_cv,
    passthrough=best_params['passthrough']
)

best_stack_clf.fit(X_train, y_train)

y_test_pred = best_stack_clf.predict(X_test)

test_acc = accuracy_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred, average='macro', zero_division=0)
test_rec = recall_score(y_test, y_test_pred, average='macro', zero_division=0)

print("Test accuracy:", test_acc)
print("Test precision:", test_prec)
print("Test recall:", test_rec)

Test accuracy: 0.9023668639053254
Test precision: 0.8986490126063197
Test recall: 0.8896501181209431


## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

## Stacking classifier is the best model
Test accuracy: 0.9023668639053254

Test precision: 0.8986490126063197

Test recall: 0.8896501181209431

In [14]:
df_pred = pd.DataFrame({
    'target': y_test,
    'prediction': y_test_pred
})

df_pred['mistake'] = df_pred['prediction'] != df_pred['target']

res = df_pred.groupby('target')['mistake'].mean() *100
res

target
0    25.925926
1     5.454545
2     6.666667
3     5.000000
4     9.523810
5    14.814815
6     9.859155
Name: mistake, dtype: float64

In [15]:
lab_cols = list(filter(lambda c: c.startswith("labname_"), X_test.columns))

lab_percent = dict()

for lab_col in lab_cols:
    mask = X_test[lab_col] == 1.0
    
    current_rows = df_pred.loc[mask]

    total = current_rows.shape[0]

    if total:
        mistakes = current_rows["mistake"].sum()
        percentage = 100.0 * mistakes / total
    else:
        percentage = 0.0

    lab_percent[lab_col] = percentage

for k, v in lab_percent.items():
    print(f'{k}: {v:.2f}')



labname_code_rvw: 15.38
labname_lab02: 0.00
labname_lab03: 100.00
labname_lab03s: 100.00
labname_lab05s: 33.33
labname_laba04: 25.71
labname_laba04s: 24.00
labname_laba05: 2.13
labname_laba06: 11.11
labname_laba06s: 6.67
labname_project1: 4.84


In [16]:
user_col = list(filter(lambda c: c.startswith("uid_user_"), X_test.columns))

u_percent = dict()

for u_col in user_col:
    mask = X_test[u_col] == 1.0
    
    current_rows = df_pred.loc[mask]

    total = current_rows.shape[0]

    if total:
        mistakes = current_rows["mistake"].sum()
        percentage = 100.0 * mistakes / total
    else:
        percentage = 0.0

    u_percent[u_col] = percentage


sorted_users = sorted(
    u_percent.items(),
    key=lambda pair: pair[1],
    reverse=True,
)


for i, j in sorted_users:
    print(f"{i}: {j:.2f}")



uid_user_6: 50.00
uid_user_16: 40.00
uid_user_30: 25.00
uid_user_3: 21.43
uid_user_13: 17.65
uid_user_18: 16.67
uid_user_27: 16.67
uid_user_19: 15.79
uid_user_17: 14.29
uid_user_2: 14.29
uid_user_14: 12.90
uid_user_31: 11.11
uid_user_24: 9.09
uid_user_29: 9.09
uid_user_10: 8.33
uid_user_4: 7.41
uid_user_0: 0.00
uid_user_1: 0.00
uid_user_11: 0.00
uid_user_12: 0.00
uid_user_15: 0.00
uid_user_20: 0.00
uid_user_21: 0.00
uid_user_22: 0.00
uid_user_23: 0.00
uid_user_25: 0.00
uid_user_26: 0.00
uid_user_28: 0.00
uid_user_7: 0.00
uid_user_8: 0.00


In [17]:
dump(best_stack_clf, '03_ensembles.joblib')

['03_ensembles.joblib']